In [1]:
# import sys
# package_path = "/localscratch/mlobo6/spadecoder/datasets/"
# if package_path not in sys.path:
#     sys.path.append(package_path)
# from spadecoder import *

## visualize spots + pie plots of results 


import pickle


import scanpy as sc

import pandas as pd



import matplotlib.pyplot as plt
from matplotlib import colors
# color_map
# sc.settings.set_figure_params(dpi=120)

plt.rcParams['figure.figsize']=(8,8) #rescale figures
# sc.settings.verbosity = 3

sc.set_figure_params(scanpy=True, dpi_save=400,dpi=150)

# plt.rcParams["font.family"] = "Arial"
plt.rcParams['pdf.fonttype'] = 42



import numpy as np 



In [2]:
figdir = '../fig2_supp5_fixed_vs_variable_neigh/'

In [3]:

def eval_perspot(gnd_truth,deconv_res):
  
    gnd_truth = gnd_truth.T
    
    deconv_res.columns = gnd_truth.columns # keeping cell ids consistent 
    assert set(deconv_res.index) == set(gnd_truth.index), "the ground truth and deconv results have different cell types"
    deconv_res = deconv_res.loc[gnd_truth.index,]

    # pearson corr 
    pearson_cor = gnd_truth.corrwith(deconv_res, axis = 0, method='pearson') # if not doing rank correlation, normalization will matter 
    # avg_corr_pe = pearson_cor.mean()


    # dom ctype - True / False 
    pred_dom_ct = deconv_res.idxmax()
    gt_dom_ct = gnd_truth.idxmax()
    is_correct_dom = (pred_dom_ct == gt_dom_ct).astype(int)
    # is_correct_dom


    # correlation between cell-types 
    correlations_pe = pd.DataFrame(index=deconv_res.T.columns, columns=gnd_truth.T.columns)
    for col1 in deconv_res.T.columns:
        for col2 in gnd_truth.T.columns:
            # gives nan when all 0's due to cell type not present in ref or query
            # correlations_sp.at[col1, col2] = deconv_res.T[col1].corr(gnd_truth.T[col2], method='spearman')
            correlations_pe.at[col1, col2] = deconv_res.T[col1].corr(gnd_truth.T[col2], method='pearson')
    # correlations_pe



    # euclidean dist 
    gnd_truth_norm = np.array(gnd_truth/gnd_truth.sum())    
    deconv_res = np.array(deconv_res/deconv_res.sum())
    # orig_rmse = np.sqrt(((gnd_truth_norm - deconv_res) ** 2).sum())
    cell_rmse = np.sqrt(((gnd_truth_norm - deconv_res) ** 2).sum(axis=0))



    per_cell_metrics = pd.DataFrame(index=gnd_truth.columns,columns=['pearson_cor','correct_dom','cell_rmse'])

    per_cell_metrics['pearson_cor'] = pearson_cor
    per_cell_metrics['correct_dom'] = is_correct_dom
    per_cell_metrics['cell_rmse'] = cell_rmse

    return per_cell_metrics, correlations_pe


In [4]:
import os 

spa_celltype_key='cell_type'

dataset = '9'

scrna_cluster_key = "cell_type"



dataset = 'Haviv2025'
dataset_full = 'dataset9_xenuim'
pickle_path = '../../datasets/' + dataset_full + '/results/simulations/pickles/'
N = 50
nneigh = 10
nbdswaps = 2
par_lambda_curr = 0.1
# par_eta1_curr = 10.0
kernel3d_bw_slices_curr = 8
bandwidth_curr = 0.01
n_spatial_neigh_curr = 10
mode_nbd = 'variabletranscr'
gtalign = False # if True, use GT alignment 
aligntool = 'moscot'
key_name = 'linear_0'
num_augment = '20'
mode_nbd = 'variabletranscr'
augment = True
mode = 'multislice'
sc_type = 'sc'
suffix = '_norm1knolog'



celltype_classes = ['Astrocyte',
 'Endothelial',
 'Excitatory Neuron',
 'Fibroblast',
 'Immune',
 'Inhibitory Neuron',
 'Oligodendrocyte',
 'Tumor']

celltype_colors = ['#1f77b4',
 '#ff7f0e',
 '#279e68',
 '#d62728',
 '#aa40fc',
 '#8c564b',
 '#e377c2',
 '#b5bd61']


deconv_dir = figdir + dataset + '/deconv/'
adata_sc = sc.read('../../datasets/' + dataset_full +   "/data/scrna_ref_norm1knolog.h5ad")


adata_spa_path = pickle_path + 'multi_slice_simulated_' + 'sptsz_' + str(N)  + '_nneigh_' + str(nneigh) + '_nbdswaps_' + str(nbdswaps) + suffix + '.pickle' # '_old.pickle'        
# read spatial file 
if not os.path.exists(adata_spa_path):
    sys.exit() 
with open(adata_spa_path, 'rb') as handle:
    adata_spa = pickle.load(handle)


key_name = 'linear_0'

result_metric = ['orig_rmse',   'avg_corr_pe','avg_jsd']




all_algos = {}


partesting_str =     '_partesting_' + 'parlambda_' + str(par_lambda_curr) +   '_parbw_' + str(bandwidth_curr) +  '_nbdtype_' + mode_nbd + '_gtalign_' + str(gtalign) + '_' + str(augment) + '_kernel3d_bw_slices_' + str(kernel3d_bw_slices_curr) + '_num_augment_' + str(num_augment) + '_realalign_' + str(aligntool) 
res_file = 'deconv_' + mode + '_' + mode_nbd + '_sptsz_' + str(N)  + '_nneigh_' + str(nneigh) + '_nbdswaps_' + str(nbdswaps) + partesting_str +'_sc_sim.pickle'
print(res_file)
with open(deconv_dir + res_file, 'rb') as handle:
    metrics_sc_ms = pickle.load(handle)
all_algos['SpaDecoder'] = metrics_sc_ms

mode_nbd = 'fixed'
for spatialneigh  in [10, 20, 30]:
    mode_nbd_curr = mode_nbd + '_nspatialneigh_' + str(spatialneigh)
    partesting_str =     '_partesting_' + 'parlambda_' + str(par_lambda_curr) +   '_parbw_' + str(bandwidth_curr) +  '_nbdtype_' + mode_nbd + '_gtalign_' + str(gtalign) + '_' + str(augment) + '_kernel3d_bw_slices_' + str(kernel3d_bw_slices_curr) + '_num_augment_' + str(num_augment) + '_realalign_' + str(aligntool) 
    res_file = 'deconv_' + mode + '_' + mode_nbd_curr + '_sptsz_' + str(N)  + '_nneigh_' + str(nneigh) + '_nbdswaps_' + str(nbdswaps) + partesting_str +'_' + sc_type + '_sim.pickle'
    print(res_file)
    with open(deconv_dir +  res_file, 'rb') as handle:
        metrics_sc_ms = pickle.load(handle)
    all_algos['SpaDecoder-Fixed-' + str(spatialneigh)] = metrics_sc_ms









deconv_multislice_variabletranscr_sptsz_50_nneigh_10_nbdswaps_2_partesting_parlambda_0.1_parbw_0.01_nbdtype_variabletranscr_gtalign_False_True_kernel3d_bw_slices_8_num_augment_20_realalign_moscot_sc_sim.pickle
deconv_multislice_fixed_nspatialneigh_10_sptsz_50_nneigh_10_nbdswaps_2_partesting_parlambda_0.1_parbw_0.01_nbdtype_fixed_gtalign_False_True_kernel3d_bw_slices_8_num_augment_20_realalign_moscot_sc_sim.pickle
deconv_multislice_fixed_nspatialneigh_20_sptsz_50_nneigh_10_nbdswaps_2_partesting_parlambda_0.1_parbw_0.01_nbdtype_fixed_gtalign_False_True_kernel3d_bw_slices_8_num_augment_20_realalign_moscot_sc_sim.pickle
deconv_multislice_fixed_nspatialneigh_30_sptsz_50_nneigh_10_nbdswaps_2_partesting_parlambda_0.1_parbw_0.01_nbdtype_fixed_gtalign_False_True_kernel3d_bw_slices_8_num_augment_20_realalign_moscot_sc_sim.pickle


/tmp/ipykernel_949774/3499817298.py:63: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  adata_spa = pickle.load(handle)
/tmp/ipykernel_949774/3499817298.py:80: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is

In [5]:
all_algos['GroundTruth'] = adata_spa.copy()

In [6]:
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt

key_name = 'linear_0'



#     with PdfPages(output_pdf) as pdf:
metrics_dict = {}
corr_dict = {}


for realidx in all_algos['SpaDecoder'].keys(): # real slice 
    for simidx in all_algos['SpaDecoder'][realidx].keys(): # sim slice 
        adata_dict = {}
        gnd_truth = all_algos['GroundTruth'][key_name][simidx][realidx].copy()
        tmp_set = set(celltype_classes) - set(gnd_truth.obs.columns)

        if len(tmp_set) != 0:
            for entry_tmp in tmp_set:
                gnd_truth.obs[entry_tmp] = 0.0
                gnd_truth.obs = gnd_truth.obs[celltype_classes]

        for algo_name in all_algos:
            if algo_name != 'GroundTruth':
                deconv_res = all_algos[algo_name][realidx][simidx].copy()
                samp_name = algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)
                metrics_dict[samp_name], corr_dict[samp_name] = eval_perspot(gnd_truth.obs,deconv_res)



                

In [26]:
import os
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

def viz_slice_spots_metrics_individual(spacoord_x, spacoord_y, plot_df, metric, 
                                       outdir=figdir, cmap='viridis', dpi=1200):
    """
    Save individual scatter plots for each metric column in plot_df.

    Parameters:
      spacoord_x : array-like
          x coordinates common to all scatter plots.
      spacoord_y : array-like
          y coordinates common to all scatter plots.
      plot_df : pandas.DataFrame
          Each column represents values for coloring the points in one subplot.
      metric : str
          Label for the colorbar (e.g. 'Expression', 'Score').
      outdir : str
          Directory to save individual plots.
      cmap : str
          Colormap for scatter plots.
      dpi : int
          Resolution (dots per inch) for saved figures.
    """
    os.makedirs(outdir, exist_ok=True)

    # Consistent color scale across all metrics
    vmin = plot_df.min().min()
    vmax = plot_df.max().max()

    for col in plot_df.columns:
        fig, ax = plt.subplots(figsize=(6, 6))
        sc = ax.scatter(spacoord_x, spacoord_y,
                        c=plot_df[col].values,
                        cmap=cmap,
                        s=50,
                        vmin=vmin, vmax=vmax,
                        edgecolor='black', linewidth=0.3)
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title(col, fontsize=16)

        # Colorbar
        norm = Normalize(vmin=vmin, vmax=vmax)
        sm = ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, orientation='vertical', fraction=0.046, pad=0.04)
        cbar.set_label(metric, fontsize=12)

        plt.tight_layout()
        fig.patch.set_facecolor('white')

        # Save figure
        save_path = os.path.join(outdir, f"{metric}_{col}_diff_from_spadecoder.pdf")
        plt.savefig(save_path, dpi=dpi, bbox_inches='tight', facecolor='white')
        plt.close(fig)

        print(f"✅ Saved: {save_path}")


In [43]:
import os
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.colors import ListedColormap


def viz_slice_spots_metrics_individual_dom_ct_diff(spacoord_x, spacoord_y, plot_df, metric, #  cmap='viridis',
                                       outdir=figdir, dpi=1200):
    """
    Save individual scatter plots for each metric column in plot_df.

    Parameters:
      spacoord_x : array-like
          x coordinates common to all scatter plots.
      spacoord_y : array-like
          y coordinates common to all scatter plots.
      plot_df : pandas.DataFrame
          Each column represents values for coloring the points in one subplot.
      metric : str
          Label for the colorbar (e.g. 'Expression', 'Score').
      outdir : str
          Directory to save individual plots.
      cmap : str
          Colormap for scatter plots.
      dpi : int
          Resolution (dots per inch) for saved figures.
    """
    os.makedirs(outdir, exist_ok=True)

    # Consistent color scale across all metrics
    # vmin = plot_df.min().min()
    # vmax = plot_df.max().max()

    color_map = {-1: "blue", 0: "lightgray", 1: "red"}
    
    for col in plot_df.columns:
        colors = plot_df[col].map(color_map)
        fig, ax = plt.subplots(figsize=(6, 6))
        sc = ax.scatter(spacoord_x, spacoord_y, #c=colors,
                        c=colors,
                        #cmap='cmap',
                        s=50,
                        #vmin=vmin, vmax=vmax,
                        edgecolor='black', linewidth=0.3)
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title(col, fontsize=16)

        # Colorbar
        # norm = Normalize(vmin=vmin, vmax=vmax)
        # sm = ScalarMappable(cmap=cmap, norm=norm)
        # sm.set_array([])
        # cbar = fig.colorbar(sm, ax=ax, orientation='vertical', fraction=0.046, pad=0.04)
        # cbar.set_label(metric, fontsize=12)

        # Add legend manually
        # handles = [plt.Line2D([0], [0], marker='o', color='w', 
        #                       label=lbl, markersize=10, 
        #                       markerfacecolor=clr, markeredgecolor='black')
        #            for lbl, clr in color_map.items()]
        # ax.legend(handles, ['-1', '0', '+1'], title=metric, loc='upper right', frameon=False)



        plt.tight_layout()
        fig.patch.set_facecolor('white')

        # Save figure
        save_path = os.path.join(outdir, f"{metric}_{col}_diff_from_spadecoder.pdf")
        plt.savefig(save_path, dpi=dpi, bbox_inches='tight', facecolor='white')
        plt.close(fig)

        print(f"✅ Saved: {save_path}")


In [ ]:
## run viz code 
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt


key_name = 'linear_0'

for metric in ['pearson_cor','correct_dom','cell_rmse']:

    #output_pdf = figdir + "./viz_slices_deconv_output_dataset_" + dataset + "_metric_" + metric + ".pdf"

    if metric != 'correct_dom':
        continue
    
    # with PdfPages(output_pdf) as pdf:

    for realidx in all_algos["SpaDecoder"].keys(): # real slice 
        for simidx in all_algos["SpaDecoder"][realidx].keys(): # sim slice 
            plot_df = pd.DataFrame(index=all_algos['GroundTruth'][key_name][simidx][realidx].obs.index)
            spacoord_x = all_algos['GroundTruth'][key_name][simidx][realidx].obsm['spatial'][:,0]
            spacoord_y = all_algos['GroundTruth'][key_name][simidx][realidx].obsm['spatial'][:,1]
            for algo_name in all_algos:
                if algo_name != 'GroundTruth':
                    samp_name = algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)
                    plot_df[samp_name] = metrics_dict[samp_name][metric].copy()
                    
                    
                    plot_df[samp_name] = metrics_dict['SpaDecoderSample:'+str(realidx)+'Sim:'+str(simidx)]['correct_dom'] - plot_df[samp_name]
                    # print(adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)])
            # fig = viz_slice_spots_metrics_individual(spacoord_x,spacoord_y, plot_df,metric,legend=True)
            
            fig = viz_slice_spots_metrics_individual_dom_ct_diff(spacoord_x,spacoord_y, plot_df,metric)


            #     """

            # fig = plt.gcf()

            # pdf.savefig(fig)
            
            
            # # plt.show()
            # plt.close(fig)
            

                

In [ ]:
## run viz code 
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt


key_name = 'linear_0'

for metric in ['pearson_cor','correct_dom','cell_rmse']:

    #output_pdf = figdir + "./viz_slices_deconv_output_dataset_" + dataset + "_metric_" + metric + ".pdf"


    #with PdfPages(output_pdf) as pdf:

    for realidx in all_algos["SpaDecoder"].keys(): # real slice 
        for simidx in all_algos["SpaDecoder"][realidx].keys(): # sim slice 
            plot_df = pd.DataFrame(index=all_algos['GroundTruth'][key_name][simidx][realidx].obs.index)
            spacoord_x = all_algos['GroundTruth'][key_name][simidx][realidx].obsm['spatial'][:,0]
            spacoord_y = all_algos['GroundTruth'][key_name][simidx][realidx].obsm['spatial'][:,1]
            for algo_name in all_algos:
                if algo_name != 'GroundTruth':
                    samp_name = algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)
                    plot_df[samp_name] = metrics_dict[samp_name][metric].copy()

                    if metric == 'correct_dom':
                        plot_df[samp_name] = metrics_dict['SpaDecoderSample:'+str(realidx)+'Sim:'+str(simidx)]['correct_dom'] - plot_df[samp_name]
                    # print(adata_dict[algo_name + 'Sample:' + str(realidx) + 'Sim:' + str(simidx)])
            # fig = viz_slice_spots_metrics_individual(spacoord_x,spacoord_y, plot_df,metric,legend=True)
            
            fig = viz_slice_spots_metrics_individual(spacoord_x,spacoord_y, plot_df,metric)


            #     """

            # fig = plt.gcf()

            # pdf.savefig(fig)
            
            
            # # plt.show()
            # plt.close(fig)
                

                

✅ Saved: ../fig2_supp5_fixed_vs_variable_neigh/pearson_cor_SpaDecoderSample:0Sim:0_diff_from_spadecoder.pdf
✅ Saved: ../fig2_supp5_fixed_vs_variable_neigh/pearson_cor_SpaDecoder-Fixed-10Sample:0Sim:0_diff_from_spadecoder.pdf
✅ Saved: ../fig2_supp5_fixed_vs_variable_neigh/pearson_cor_SpaDecoder-Fixed-20Sample:0Sim:0_diff_from_spadecoder.pdf
✅ Saved: ../fig2_supp5_fixed_vs_variable_neigh/pearson_cor_SpaDecoder-Fixed-30Sample:0Sim:0_diff_from_spadecoder.pdf
✅ Saved: ../fig2_supp5_fixed_vs_variable_neigh/pearson_cor_SpaDecoderSample:0Sim:1_diff_from_spadecoder.pdf
✅ Saved: ../fig2_supp5_fixed_vs_variable_neigh/pearson_cor_SpaDecoder-Fixed-10Sample:0Sim:1_diff_from_spadecoder.pdf
✅ Saved: ../fig2_supp5_fixed_vs_variable_neigh/pearson_cor_SpaDecoder-Fixed-20Sample:0Sim:1_diff_from_spadecoder.pdf
✅ Saved: ../fig2_supp5_fixed_vs_variable_neigh/pearson_cor_SpaDecoder-Fixed-30Sample:0Sim:1_diff_from_spadecoder.pdf
✅ Saved: ../fig2_supp5_fixed_vs_variable_neigh/pearson_cor_SpaDecoderSample:0Sim:2